# Annotating scans for the text-analysis tasks

This notebook lets you step through the scans of a single inventory number and label them for
the four text-analysis tasks, using the ground-truth format in
`archival_structures.datasets.annotations`:

1. **Opening** — is this scan a two-page spread, and if so, where's the split? Pre-filled with a
   suggestion from `archival_structures.analysis.opening_detection`, which you can correct.
2. **Page layout** — a free-text label for the page's overall layout type (e.g. `two_column_body`).
3. **Lines** — a label per text line (e.g. `body`, `closing`, `marginalium`). Tick the checkboxes
   of the lines you want to label (matching the numbers drawn on the thumbnail), pick a label from
   the dropdown, and click **Assign label** to apply it to all ticked lines at once — handy for
   labelling a whole run of body-text lines in one go.

Cross-page document elements (task 4) aren't part of this per-scan flow — see the short section
at the end for how to record those directly with the `annotations` API.

Labels are saved to `data/annotations/<institute_id>/<archive_id>/<inventory_num_id>/annotations-<scan_id>.json`
(one file per scan) as soon as you click **Save**. Re-opening a scan you already labelled loads
your previous answers instead of the suggested defaults.

**You need `ipywidgets` installed and enabled in this Jupyter environment for the interactive
controls below to render** (`pip install ipywidgets`, then restart the kernel).

In [4]:
from pathlib import Path

from PIL import Image, ImageDraw, ImageFont
import ipywidgets as widgets
from IPython.display import display, clear_output
import pagexml.parser as pagexml_parser

from archival_structures.image.image_base import make_selection
from archival_structures.model.image import Box
from archival_structures.analysis.opening_detection import get_opening_features
from archival_structures.datasets.annotations import (
    OpeningLabel, ScanAnnotation, new_scan_annotation,
    load_scan_annotation, save_scan_annotation, scan_annotation_path,
)

# --- configure which inventory number to annotate ---
INSTITUTE = 'NL-AsnDA'
ARCHIVE = 'NL-AsnDA_0114.11'
INVENTORY_NUM = 'NL-AsnDA_0114.11_1'

PAGEXML_DIR = Path('../../data/PageXML') / INSTITUTE / ARCHIVE / INVENTORY_NUM
IMAGE_DIR = Path('../../data/thumbs') / INSTITUTE / ARCHIVE / INVENTORY_NUM

# the line-type vocabulary offered in the dropdowns -- edit freely as you discover more types
LINE_TYPE_CHOICES = [
    '', 'body', 'heading', 'closing', 'marginalium', 'signature', 'catchword', 'other',
    'table_cell', 'table_row', 'running_text', 'para_start', 'para_mid', 'para_end'
]

assert PAGEXML_DIR.exists(), PAGEXML_DIR
assert IMAGE_DIR.exists(), IMAGE_DIR

## Plain helper functions

These don't depend on `ipywidgets`, so they can be (and were) tested by just calling them directly.

In [5]:
def scan_xml_paths() -> list[Path]:
    """All PageXML files for the configured inventory number, in filename (= page sequence) order."""
    return sorted(PAGEXML_DIR.glob('*.xml'))


def thumb_path_for(xml_path: Path) -> Path:
    return IMAGE_DIR / f"thumb-width_300-scan-{xml_path.stem}.jp2.png"


def ordered_lines(scan):
    """Text lines of `scan`, roughly in reading order (top-to-bottom, then left-to-right).
    This is a simple sort for browsing convenience, not an authoritative reading order -- lines
    in side-by-side columns at the same height may come out in either order."""
    return sorted(scan.get_lines(), key=lambda line: (line.coords.top, line.coords.left))


def line_box(line) -> Box:
    return Box(line.coords.left, line.coords.top, line.coords.width, line.coords.height, label=line.id)


def load_or_init_annotation(scan) -> ScanAnnotation:
    """Load this scan's saved annotation, or build a fresh skeleton with an opening-detection
    suggestion if it hasn't been annotated yet."""
    path = scan_annotation_path(INSTITUTE, ARCHIVE, INVENTORY_NUM, scan.id)
    if path.exists():
        return load_scan_annotation(path)
    annotation = new_scan_annotation(scan)
    features = get_opening_features(scan)
    annotation.opening = OpeningLabel(is_opening=features.is_opening, separation_x=features.separation_point.x)
    return annotation


def draw_numbered_lines(thumb_image: Image.Image, image_sel, lines) -> Image.Image:
    """Draw each line's bounding box on a copy of the thumbnail, numbered 1..N in the same order
    as `lines`, so the numbers can be matched to the line checkboxes built from the same list."""
    annotated = thumb_image.copy()
    draw = ImageDraw.Draw(annotated)
    for line_no, line in enumerate(lines, start=1):
        box = image_sel.scan_to_thumb.apply(line_box(line))
        draw.rectangle([box.x, box.y, box.x + box.w, box.y + box.h], outline='red', width=1)
        draw.text((box.x, max(0, box.y - 9)), str(line_no), fill='blue')
    return annotated

## The annotation app

Run the cell below, then use **Previous**/**Next** to move between scans and **Save** to write
out the current scan's labels before moving on (moving to another scan does not auto-save).

In [6]:
class ScanAnnotationApp:

    def __init__(self):
        self.xml_paths = scan_xml_paths()
        if not self.xml_paths:
            raise FileNotFoundError(f"no PageXML files found under {PAGEXML_DIR}")
        self.index = 0
        self.line_checkboxes: dict[str, widgets.Checkbox] = {}
        self.line_label_displays: dict[str, widgets.Label] = {}
        self.line_labels: dict[str, str | None] = {}

        self.prev_button = widgets.Button(description='← Previous')
        self.next_button = widgets.Button(description='Next →')
        self.save_button = widgets.Button(description='Save', button_style='success')
        self.status_label = widgets.Label()
        self.prev_button.on_click(self._go_prev)
        self.next_button.on_click(self._go_next)
        self.save_button.on_click(self._save)

        self.select_all_button = widgets.Button(description='Select all')
        self.deselect_all_button = widgets.Button(description='Deselect all')
        self.label_choice_dropdown = widgets.Dropdown(options=LINE_TYPE_CHOICES, description='label')
        self.assign_label_button = widgets.Button(description='Assign label', button_style='info')
        self.select_all_button.on_click(self._select_all)
        self.deselect_all_button.on_click(self._deselect_all)
        self.assign_label_button.on_click(self._assign_label)

        self.output = widgets.Output()
        nav = widgets.HBox([self.prev_button, self.next_button, self.save_button, self.status_label])
        display(nav, self.output)
        self.render()

    def render(self):
        xml_path = self.xml_paths[self.index]
        thumb_path = thumb_path_for(xml_path)
        scan = pagexml_parser.parse_pagexml_file(str(xml_path))
        self.scan = scan
        self.lines = ordered_lines(scan)
        self.annotation = load_or_init_annotation(scan)
        self.line_labels = dict(self.annotation.lines)
        self.status_label.value = ''

        with self.output:
            clear_output(wait=True)
            print(f"[{self.index + 1}/{len(self.xml_paths)}] {scan.id}")

            if thumb_path.exists():
                image_sel = make_selection(scan.coords.w, scan.coords.h, str(thumb_path))
                thumb_image = Image.open(thumb_path).convert('RGB')
                display(draw_numbered_lines(thumb_image, image_sel, self.lines))
            else:
                print(f"(no thumbnail found at {thumb_path})")

            self.is_opening_checkbox = widgets.Checkbox(
                value=bool(self.annotation.opening.is_opening) if self.annotation.opening else False,
                description='is_opening')
            self.separation_x_box = widgets.FloatText(
                value=self.annotation.opening.separation_x if self.annotation.opening else 0.0,
                description='separation_x')
            self.page_layout_box = widgets.Text(
                value=self.annotation.page_layout or '', description='page_layout')
            display(widgets.HBox([self.is_opening_checkbox, self.separation_x_box]))
            display(self.page_layout_box)

            print(f"\n{len(self.lines)} lines (numbers match the image above).")
            print("Tick the lines you want to label, pick a label, then click 'Assign label'.")
            display(widgets.HBox([self.select_all_button, self.deselect_all_button,
                                  self.label_choice_dropdown, self.assign_label_button]))

            self.line_checkboxes = {}
            self.line_label_displays = {}
            line_rows = []
            row_layout = widgets.Layout(flex='0 0 auto')
            for line_no, line in enumerate(self.lines, start=1):
                checkbox = widgets.Checkbox(value=False, description=f"#{line_no} ({line.id})",
                                            indent=False, layout=widgets.Layout(width='220px'))
                label_display = widgets.Label(value=self.line_labels.get(line.id) or '(unlabeled)')
                self.line_checkboxes[line.id] = checkbox
                self.line_label_displays[line.id] = label_display
                # flex='0 0 auto' on each row stops the fixed-height VBox below from shrinking
                # rows to fit -- without it, flexbox compresses every row to squeeze them into the
                # box instead of scrolling past them at their natural height, so rows visually
                # collapse and overlap rather than stacking properly.
                line_rows.append(widgets.HBox([checkbox, label_display], layout=row_layout))
            # NB: `overflow_y` was removed from ipywidgets' Layout in v8 (replaced by the single
            # `overflow` property) -- passing it is silently swallowed (just a DeprecationWarning,
            # no error), so the box never actually got scrolling behaviour. Use `overflow='auto'`.
            display(widgets.VBox(line_rows, layout=widgets.Layout(
                height='400px', overflow='auto', flex_flow='column', align_items='flex-start')))

    def _select_all(self, _button):
        for checkbox in self.line_checkboxes.values():
            checkbox.value = True

    def _deselect_all(self, _button):
        for checkbox in self.line_checkboxes.values():
            checkbox.value = False

    def _assign_label(self, _button):
        label = self.label_choice_dropdown.value or None
        for line_id, checkbox in self.line_checkboxes.items():
            if not checkbox.value:
                continue
            self.line_labels[line_id] = label
            self.line_label_displays[line_id].value = label or '(unlabeled)'
            checkbox.value = False

    def _collect_annotation(self) -> ScanAnnotation:
        opening = OpeningLabel(is_opening=self.is_opening_checkbox.value,
                               separation_x=self.separation_x_box.value)
        return ScanAnnotation(scan_id=self.scan.id, opening=opening,
                              page_layout=self.page_layout_box.value or None,
                              lines=dict(self.line_labels))

    def _save(self, _button):
        annotation = self._collect_annotation()
        path = scan_annotation_path(INSTITUTE, ARCHIVE, INVENTORY_NUM, self.scan.id)
        save_scan_annotation(annotation, path)
        self.status_label.value = f"saved to {path}"

    def _go_prev(self, _button):
        self.index = max(0, self.index - 1)
        self.render()

    def _go_next(self, _button):
        self.index = min(len(self.xml_paths) - 1, self.index + 1)
        self.render()


app = ScanAnnotationApp()

Output()

## Recording cross-page elements (task 4)

Document elements that span more than one scan don't fit the per-scan flow above. Build and
append them directly with the `annotations` API instead -- run this after you've identified, in
the per-scan view, the line ids on either side of the page break that belong to the same element.

In [ ]:
from archival_structures.datasets.annotations import Element, ElementSpan, elements_path, load_elements, save_elements

path = elements_path(INSTITUTE, ARCHIVE, INVENTORY_NUM)
existing = load_elements(path) if path.exists() else []

# example -- edit scan_id/line_ids to match a real cross-page element you've spotted, then run:
# new_element = Element(element_type='closing', spans=[
#     ElementSpan(scan_id='NL-AsnDA_0114.11_1_0004.jpg', line_ids=['r2l10', 'r2l11']),
#     ElementSpan(scan_id='NL-AsnDA_0114.11_1_0005.jpg', line_ids=['r1l1']),
# ])
# existing.append(new_element)
# save_elements(existing, path)

print(f"{len(existing)} elements recorded so far at {path}")